<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 16 · Credit Spreads, Ratings, and Default Risk

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch16_credit_spreads_ratings_default_risk.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This notebook makes the credit-risk language of Chapter 16 computable. You will

- decompose a quoted credit spread into expected loss and a spread pickup,
- compute expected loss as default probability times loss given default, in basis points of face value,
- read a synthetic issuer panel with ratings, spreads, spread duration, and recovery,
- separate real-world default probabilities used for loss analysis from risk-neutral ones implied by market prices, and
- check rating-bucket commentary against Python-generated tables before repeating it.

Every idea is verified on the frozen teaching data in the companion repository, so you can reproduce each number yourself.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The cell below runs the Chapter 16 script. It loads the synthetic issuer panel, computes each issuer's expected loss in basis points as default probability times loss given default, and subtracts it from the quoted spread to get the spread pickup. It then averages spreads and expected loss by rating bucket and closes with the panel's first-order loss from a 50 bp parallel spread widening.

In [ ]:
SCRIPT = ROOT / "code" / "ch16_credit_spreads_ratings_default_risk.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

- Confirm the expected-loss convention: default probability times (1 minus recovery), in basis points of face value, for example 22.0 bp for Alpha Utilities at 0.4% PD and 45% recovery.
- Check that every pickup_bp is positive: quoted spreads of 115 to 310 bp stand well above expected losses of 22.0 to 195.0 bp, so a spread is not equal to expected loss.
- Compare the rating summary: the BB bucket averages 310 bp of spread against 195.0 bp of expected loss, a wider gap in absolute terms than the A bucket's 125 bp against 29.0 bp.
- Read the 50 bp widening loss of 424,750 as a first-order CS01 estimate: market value times spread duration times 50/10,000, with no convexity or default interaction.
- Note that the panel's PDs are real-world inputs, not risk-neutral hazards implied by market prices.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch16_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: From Hazard Rate to Expected Loss and Spread Pickup

Three illustrative issuers carry default intensities (hazard rates) of 1%, 2%, and 3% per year, with recovery of 40% of face value. Under a constant hazard, the one-year default probability is one minus the survival probability, and expected loss in basis points is that probability times loss given default. The shortcut column applies the reduced-form approximation spread equal to hazard times LGD, which tracks expected loss closely at short horizons. The quoted spreads of 115 to 310 bp exceed expected loss by a pickup that compensates for risk premia, liquidity, and uncertainty, not just expected default.

In [ ]:
import numpy as np
import pandas as pd

hazard = np.array([0.010, 0.020, 0.030])  # intensity per year
recovery = 0.40  # recovered fraction of face in default
quoted = np.array([115.0, 185.0, 310.0])  # quoted spread in bp
lgd = 1.0 - recovery  # loss given default
one_year_pd = 1.0 - np.exp(-hazard)  # one-year default probability
table = pd.DataFrame({
    "hazard": hazard,
    "one_year_pd": one_year_pd.round(4),
    "expected_loss_bp": (one_year_pd * lgd * 10_000).round(1),
    "shortcut_bp": (hazard * lgd * 10_000).round(1),
    "quoted_spread_bp": quoted,
    "pickup_bp": (quoted - one_year_pd * lgd * 10_000).round(1),
})
table

### What to observe

The shortcut tracks expected loss closely at one year because `1 - exp(-hazard)` is nearly linear for small hazards: 60.0, 120.0, and 180.0 bp against exact values of 59.7, 118.8, and 177.3 bp. The quoted spreads exceed both, leaving pickups of 55.3, 66.2, and 132.7 bp. The pickup is largest for the riskiest issuer, where premium and liquidity compensation are greatest.

## Exercises

1. **Expected loss by hand.** Compute expected loss in basis points for a PD of 2% and recovery of 40%. *Hint:* 0.02 times 0.60 times 10,000.
2. **Rating-bucket comparison.** Rank the three rating buckets by spread and by expected loss, and explain in two sentences why the pickup differs across buckets.
3. **Spread shock.** Recompute the 50 bp widening loss with a 100 bp shock and confirm the loss scales linearly. *Hint:* panel CS01 is the sum of market value times spread duration over 10,000.
4. **Transition matrix.** Load `data/ch16_rating_transition_matrix.csv` and compute the one-year probability of a BBB issuer ending below BBB, including default.
5. **Spread panel.** Load `data/ch16_credit_spread_panel.csv` and compute each bucket's cumulative spread change over the sample in basis points.
6. **Spread is not expected loss.** Explain why Gamma Retail's 310 bp spread does not imply a 3.1% annual default probability. *Hint:* the spread also prices risk premia, liquidity, and taxes.

For more detail, see Chapter 16 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">